# Train ResNet on MNIST-rot

This notebook trains a standard `ResNet` on the rotated-MNIST dataset using the shared `dataloaders.py` module.

Update the config cell if you want to change model width, epochs, batch size, or device.

In [ ]:
from pathlib import Path
import sys

import torch
from torch import nn
from tqdm.auto import tqdm

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "dataloaders.py").exists():
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / "dataloaders.py").exists():
    raise FileNotFoundError("Could not locate repo root containing dataloaders.py")

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from dataloaders import create_rotated_mnist_dataloaders
from resnet import ResNet

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

epochs = 5
batch_size = 128
eval_batch_size = 256
learning_rate = 1e-3
weight_decay = 1e-4
n_blocks = 2
channel_dims = [32, 64, 128]
num_workers = 0

train_loader, test_loader = create_rotated_mnist_dataloaders(
    train_batch_size=batch_size,
    eval_batch_size=eval_batch_size,
    num_workers=num_workers,
)

model = ResNet(
    n_blocks=n_blocks,
    in_channels=1,
    num_classes=10,
    channel_dims=channel_dims,
).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=learning_rate,
    weight_decay=weight_decay,
)

print(f"Using device: {device}")
print(f"Train examples: {len(train_loader.dataset)}")
print(f"Test examples: {len(test_loader.dataset)}")

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    total_correct = 0
    total_examples = 0

    for images, labels in tqdm(loader, desc="train", leave=False):
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad(set_to_none=True)
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        batch_size = labels.size(0)
        total_examples += batch_size
        total_loss += loss.item() * batch_size
        total_correct += (logits.argmax(dim=1) == labels).sum().item()

    return total_loss / total_examples, total_correct / total_examples


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_examples = 0

    for images, labels in tqdm(loader, desc="eval", leave=False):
        images = images.to(device)
        labels = labels.to(device)

        logits = model(images)
        loss = criterion(logits, labels)

        batch_size = labels.size(0)
        total_examples += batch_size
        total_loss += loss.item() * batch_size
        total_correct += (logits.argmax(dim=1) == labels).sum().item()

    return total_loss / total_examples, total_correct / total_examples

In [ ]:
history = []

for epoch in range(1, epochs + 1):
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)
    test_loss, test_acc = evaluate(model, test_loader, criterion, device)

    metrics = {
        "epoch": epoch,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "test_loss": test_loss,
        "test_acc": test_acc,
    }
    history.append(metrics)
    print(metrics)

history